In [ ]:
%load_ext autoreload
%autoreload 2

# Primary synthetic datasets

In [ ]:
import os
from geopandas import read_parquet, GeoDataFrame, GeoSeries,points_from_xy
import geopandas as gpd
import pyarrow.parquet as pq
import shapely
import numpy as np
import scanpy as sc
import pandas as pd
import numpy as np
from shapely import Point, Polygon, distance
from scipy.spatial.distance import cdist
from scipy.sparse import csc_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import argparse
import adjustText
from tqdm import tqdm

MERSCOPE_DATADIR = '../data/merscope_hcc1'

In [ ]:
def calculate_mask_distance(
    centroid_dist: pd.DataFrame,
    cell_coords: pd.DataFrame,
    max_centroid_dist: int = 15,
    min_centroid_dist: int = 0,
    cluster_col:str = 'leiden',
    x_col:str = 'x',
    y_col:str = 'y',
    geometry_col:str = 'Geometry',
    re_cal_centroid_dist:bool = False
    ):
    '''
    centroid_dist: cell by cell distance matrix by centroid location, 
    max_centroid_dist: maximal distance to consider neighbors, 
    max_centroid_dist: minimal distance to consider distant cells, 
    cell_coords: cell by cell distance matrix by centroid location, 
    centroid_dist: cell by cell distance matrix by centroid location, 
    '''
    adj = (centroid_dist > min_centroid_dist) & (centroid_dist<= max_centroid_dist)
    adj = adj.agg(
        lambda x: adj.columns[x.values].tolist(), axis=1)
    adj = pd.DataFrame(adj, columns = ['n'])
    # pandas suggest using transform, but it did not work.
    adj = adj.agg(
        lambda x: [
            y for y in x.n if cell_coords.loc[y, cluster_col]!=cell_coords.loc[x.name, cluster_col]
            ], axis=1)
    adj_nonself = adj[adj.apply(len)>0]
    adj_nonself_masks_ids = adj_nonself.explode()
    md = distance(
        cell_coords.loc[adj_nonself_masks_ids.index, geometry_col].values,
        cell_coords.loc[adj_nonself_masks_ids.values, geometry_col].values
    )
    masks_distances = pd.DataFrame(
        adj_nonself_masks_ids, columns=['neighbor_by_centroid'])
    masks_distances['mask_distance'] = md
    # recalculating the centroid distance is expensive, so it should be avoided
    if re_cal_centroid_dist:
        v1 = cell_coords.loc[masks_distances.index, [x_col, y_col]].values
        v2 = cell_coords.loc[masks_distances.neaghbor_by_centroid, [x_col, y_col]].values
        masks_distances['centroid_distance'] = np.sum((v1-v2)**2, axis=1)**0.5
    return masks_distances

def prepare_permutation_scheme(adata_obj, tx_meta):
    """
    Subset the HCC1 dataset and identify the tumor:non-tumor boundary

    Args:
        h5ad_filename (str): the filepath of the AnnData object
        geo_parquet_filename (str): the filepath of the parquet file containing cell boundary coordinates
    """
    adata_uq = adata_obj[(adata_obj.obs['center_x'] <= 6000) & ((adata_obj.obs['center_y'] >= 6000))].copy()

    cell_dist = cdist(
        adata_uq[adata_uq.obs['cell_type'] == 'Tumor_cells'].obs[['center_x', 'center_y']],
        adata_uq[adata_uq.obs['cell_type'] != 'Tumor_cells'].obs[['center_x', 'center_y']]
    )

    cell_dist = pd.DataFrame(
        cell_dist, 
        index = adata_uq[adata_uq.obs['cell_type'] == 'Tumor_cells'].obs_names, 
        columns = adata_uq[adata_uq.obs['cell_type'] != 'Tumor_cells'].obs_names
    )

    cell_meta = adata_uq.obs.copy()

    mask_distance = calculate_mask_distance(
        cell_dist,
        cell_meta,
        max_centroid_dist=30,
        x_col='X',
        y_col='Y',
        cluster_col = 'cell_type',
        geometry_col='polygon'
    )

    interface_tumor = mask_distance[mask_distance.mask_distance <= 2].index.unique()
    non_tumor_samples = adata_uq[adata_uq.obs['cell_type']!='Tumor_cells'].to_df().sample(len(interface_tumor)).index

    tx_interface = tx_meta[tx_meta['cell'].isin(interface_tumor)].copy()
    tx_interface['permuted_cell'] = pd.Series(non_tumor_samples, index=interface_tumor)[tx_interface['cell']].values
    tx_interface['permuted_celltype'] = cell_meta.loc[tx_interface['permuted_cell'],'cell_type'].values
    
    permuted_tx_interface = []
    print('Replace interface tumor cell transcripts with non tumor transcripts')
    for _,df in tqdm(tx_interface.groupby('permuted_celltype')):
        tx_ref = tx_meta[tx_meta['cell'].isin(df['permuted_cell'].unique())]
        permuted_genes = tx_ref['gene'].sample(df.shape[0], replace=True).values
        df['permuted_gene'] = permuted_genes
        permuted_tx_interface.append(df)
    permuted_tx_interface = pd.concat(permuted_tx_interface)

    sample_celltypes = adata_uq.obs.loc[non_tumor_samples,'cell_type'].astype(str).values
    # Construct the interface adata with swapped cells
    # The interface cells will have the same cell id but swapped gene expression.
    adata_interface = adata_uq[interface_tumor].copy()
    adata_interface.X = adata_uq[non_tumor_samples].layers['count']
    adata_interface.layers['Raw'] = adata_interface.X.copy()
    adata_interface.obs['cell_type'] = sample_celltypes

    # Construct the synthetic data by merging the orphan tumors adata and the synthetic interface adata
    adata_uq_others = adata_uq[~adata_uq.obs_names.isin(interface_tumor)]
    adata_synthetic = sc.concat([adata_uq_others, adata_interface])
    # Remake the metadata

    cell_meta_synthetic = cell_meta.copy()
    cell_meta_synthetic.loc[interface_tumor, 'cell_type'] = sample_celltypes

    # Identify tumor cells to add transcripts to
    # Need to recalculate tumor to non tumor cell distance as the cell identify was changed
    cell_dist_synthetic = pd.DataFrame(
        cdist(
            adata_synthetic[adata_synthetic.obs['cell_type'] == 'Tumor_cells'].obs[['center_x', 'center_y']],
            adata_synthetic[adata_synthetic.obs['cell_type'] != 'Tumor_cells'].obs[['center_x', 'center_y']]
        ),
        index = adata_synthetic[adata_synthetic.obs['cell_type'] == 'Tumor_cells'].obs_names, 
        columns = adata_synthetic[adata_synthetic.obs['cell_type'] != 'Tumor_cells'].obs_names
    )

    mask_distance_synthetic = calculate_mask_distance(
        cell_dist_synthetic,
        cell_meta_synthetic,
        max_centroid_dist=30,
        x_col='X',
        y_col='Y',
        cluster_col = 'cell_type',
        geometry_col='polygon'
    )
    
    return cell_dist_synthetic, cell_meta_synthetic, mask_distance_synthetic, adata_synthetic, permuted_tx_interface

def generate_synthetic_dataset(
    cell_dist_synthetic, cell_meta_synthetic, mask_distance_synthetic, adata_synthetic, permuted_tx_interface, outfile,
    max_noise_tx=5, n_points=1000, doublet_prob=0.2, tx2m_dist=1, deg_max_p=0.05, deg_min_effect=0.6, tumor_deg_max_p=0.05, tumor_deg_min_effect=1,
    max_pct_tumor = 0.5, min_pct_diff = 0.2, max_pct_other = 0.15, min_pct_tumor = 0.5,
    seed=None
):
    """
    Perform synthetic data generation
    Args:
        cell_dist_synthetic (pd.DataFrame): table of distances between pairs of cells
        cell_meta_synthetic (pd.DataFrame): table of cells in which to generate synthetic transcripts
        mask_distance_synthetic (pd.DataFrame): table of distances between transcripts and cell mask boundaries
        adata_synthetic (ad.AnnData): an AnnData object containing single cell data
        permuted_tx_interface (pd.DataFrame): table of permuted transcripts at the cell interface
        outfile (str): output file
        max_noise_tx (int, optional): max_noise_tx = 5 # number of transcripts per cell to swap (positive control) and add (negative control). Defaults to 5.
        n_points (int, optional): number of points to sample as potential spatial locations for positive controls. Defaults to 1000.
        doublet_prob (float, optional): fraction of tumor interface cells in which to simulate spillover. Defaults to 0.2.
        tx2m_dist (int, optional): maximum distance between transcripts and cell boundaries for swapping. Defaults to 1.
        deg_max_p (float, optional): significance value to be considered differentially expressed. Defaults to 0.05.
        deg_min_effect (float, optional): minimum effect size to be considered differentially expressed. Defaults to 0.6
        tumor_deg_max_p (float, optional): significance value to be considered differentially expressed in tumors. Defaults to 0.05.
        tumor_deg_min_effect (float, optional): minimum effect size to be considered differentially expressed in tumors. Defaults to 1.
        max_pct_tumor (float, optional): maximum expression frequency in tumor cells for non-tumor DEGs. Defaults to 0.5.
        min_pct_diff (float, optional): minimum difference in expression frequency between ingroup vs all rest. Defaults to 0.2.
        max_pct_other (float, optional): maximum expression frequency in other cells for tumor DEGs. Defaults to 0.6. Defaults to 0.15.
        min_pct_tumor (float, optional): minimum expression frequency in tumor cells for tumor DEGs. Defaults to 0.5.
        seed (int, optional): _description_. Defaults to None.

    Returns:
        pd.DataFrame: a table of transcripts metadata containing both permuted and real transcripts
    """
    if seed is not None:
        np.random.seed(seed)
    doublet_prob = 1 - doublet_prob
    
    interface_synthetic = mask_distance_synthetic[mask_distance_synthetic.mask_distance<0.5].copy()
    interface_synthetic['tumor_cell'] = interface_synthetic.index
    interface_synthetic = interface_synthetic.sort_values('mask_distance')
    # only simulated TX from one neighbor. 
    # TXs from another neighbor is conceptually equivalent and computationally a simple repeat.
    interface_synthetic.drop_duplicates('tumor_cell', inplace=True)
    # doublet level means the fraction of cell in the interaface to have at least one added TX
    print('Simulating TX points in {} cells. up to {} in each'.format(len(interface_synthetic), max_noise_tx))
    simulated_points_pos = {}
    simulated_points_neg = {}
    
    neighbors = []
    rng = np.random.default_rng()
    for tumor_cell in interface_synthetic.index:
        num_neg_tx = num_pos_tx = np.random.randint(1, max_noise_tx)
        tumor_p = cell_meta_synthetic.loc[tumor_cell,'polygon']
        points = np.array(GeoSeries(tumor_p).sample_points(size=n_points).iloc[0].geoms)
        neg_mask = np.zeros(1000, dtype=bool)
        neg_mask[rng.choice(1000,size=num_neg_tx, replace=False)] = True
        neg_points = points[neg_mask]
        simulated_points_neg[tumor_cell] = neg_points
        points = points[~neg_mask]
        if np.random.uniform(0,1,1) < doublet_prob:
            continue
        neighbor = interface_synthetic.loc[tumor_cell,'neighbor_by_centroid']
        neighbor_p = cell_meta_synthetic.loc[neighbor,'polygon']
        dist_to_neighbor = distance(
            points, np.repeat(neighbor_p,n_points - num_neg_tx)
        )
        pos_points = np.random.choice(
            points[dist_to_neighbor<tx2m_dist],
            min(num_pos_tx,(dist_to_neighbor<tx2m_dist).sum()),
            replace=False
        )
        if len(pos_points)==0:
            continue
        simulated_points_pos[tumor_cell]= pos_points
        neighbors += [neighbor] * len(pos_points)

    # Differential analysis
    print('Assign each TX point to a DEG from neighbor cell.')
    
    adata_synthetic.X = adata_synthetic.layers['count'].copy()
    adata_synthetic.raw = adata_synthetic.copy()
    sc.pp.normalize_total(adata_synthetic, target_sum=1000)
    sc.pp.log1p(adata_synthetic)
    sc.tl.rank_genes_groups(adata_synthetic, 'cell_type', pts=True, tie_correct=True, use_raw=False)
    deg = sc.get.rank_genes_groups_df(adata_synthetic, None)
    deg = deg.merge(deg[deg['group']=='Tumor_cells'][['names', 'pct_nz_group']].rename({'pct_nz_group': 'pct_nz_Tumor_cells'}, axis=1),on='names')

    # Filter the deg that can be added to synthetic target cells
    deg = deg[
        (deg['pvals_adj'] <= deg_max_p) & 
        (deg['logfoldchanges'] >= deg_min_effect) &
        (deg['pct_nz_Tumor_cells'] <= max_pct_tumor) & 
        ((deg['pct_nz_group'] - deg['pct_nz_reference']) >= min_pct_diff)
    ]
    print('# DEGs that can be synthetically added:', deg.shape[0])

    ct_deg = deg.groupby('group').names.agg(lambda x : x.tolist())

    # Identify genes selectively expressed in tumor cells
    non_tumor_genes = np.unique(ct_deg.sum())
    tumor_deg = sc.get.rank_genes_groups_df(
        adata_synthetic, 'Tumor_cells', log2fc_min=tumor_deg_min_effect, pval_cutoff=tumor_deg_max_p)
    tumor_genes = tumor_deg[
        (tumor_deg['pct_nz_reference'] <= max_pct_other) & 
        (tumor_deg['pct_nz_group'] >= min_pct_tumor)
    ].names.values
    tumor_genes = [x for x in tumor_genes if x not in non_tumor_genes]

    # Randomly assign gene symbol for each simulated transcript
    interface_synthetic['neighbor_ct'] = cell_meta_synthetic.loc[
        interface_synthetic['neighbor_by_centroid'].values, 'cell_type'].values
    interface_synthetic.loc[simulated_points_pos.keys(),'synthetic_pos'] = pd.Series(
        [*simulated_points_pos.values()]).values
    interface_synthetic.loc[simulated_points_neg.keys(),'synthetic_neg'] = pd.Series(
        [*simulated_points_neg.values()]).values
    interface_synthetic.loc[
        interface_synthetic['synthetic_pos'].notnull(), 'pos_symbol'
        ] = interface_synthetic.loc[interface_synthetic['synthetic_pos'].notnull()].apply(
        lambda x: np.random.choice(
            ct_deg.loc[x['neighbor_ct']], len(x['synthetic_pos'])
            ).tolist(),
        axis=1)
    interface_synthetic.loc[
        interface_synthetic['synthetic_neg'].notnull(), 'neg_symbol'
        ] = interface_synthetic.loc[interface_synthetic['synthetic_neg'].notnull()].apply(
        lambda x: np.random.choice(tumor_genes, len(x['synthetic_neg'])).tolist(),
        axis=1)    
    
    synthetic_txs = []
    for i in range(2):
        tx_type = ['synthetic_pos','synthetic_neg'][i]
        tx_symbol = ['pos_symbol', 'neg_symbol'][i]
        tmp = interface_synthetic.dropna(subset=tx_type)
        _syn_tx = pd.DataFrame(tmp[tx_type].explode())
        _syn_tx['gene'] = tmp[tx_symbol].explode()
        _syn_tx['cell_id'] = _syn_tx.index
        _syn_tx['num'] = 1
        if i == 0:
            _syn_tx['neighbor'] = neighbors
        _syn_tx['x'] = _syn_tx[tx_type].apply(lambda x: x.x)
        _syn_tx['y'] = _syn_tx[tx_type].apply(lambda x: x.y)
        _syn_tx['tx_type'] = tx_type
        _syn_tx = _syn_tx.rename(columns={tx_type:'point'})
        synthetic_txs.append(_syn_tx)
    synthetic_txs = pd.concat(synthetic_txs)

    for g, df in synthetic_txs.groupby('tx_type'):
        count_delta = df.groupby(['cell_id','gene']).num.count()
        count_delta = count_delta.reset_index().pivot(
            columns='gene', values='num', index='cell_id').fillna(0).astype(int)

        # Check if the added genes are indeed lowly expressed in target tumor cells
        print(
            '{} expression in target cells in original data:'.format(g), 
            adata_synthetic.raw.to_adata().to_df().loc[count_delta.index, count_delta.columns].sum().sum())
        print(
            '{} expression in target cells total synthetic original_counts:'.format(g), 
            count_delta.sum().sum())
        print(
            'random selected, equal-sized sets of genes in target cells original original_counts:', 
            adata_synthetic.raw.to_adata().to_df().loc[
                count_delta.index, 
                np.random.choice(adata_synthetic.raw.to_adata().to_df().columns, count_delta.shape[1])
                ].sum().sum())
        
    print('Saving outputs.')
    tx_meta_interface = tx_meta[tx_meta.cell.isin(adata_synthetic.obs_names)].copy()
    tx_meta_interface = tx_meta_interface.rename(
        columns={'global_x':'x', 'global_y':'y'})
    permuted_tx_interface = permuted_tx_interface.rename(
        columns={'global_x':'x', 'global_y':'y'})
    # Merge TX with permuted TX
    tx_meta_interface = tx_meta_interface[
        ~tx_meta_interface.cell.isin(permuted_tx_interface.cell.unique())]
    tx_meta_interface['tx_type'] = 'real'
    permuted_tx_interface['gene'] = permuted_tx_interface['permuted_gene']
    permuted_tx_interface['tx_type'] = 'permuted'
    tx_meta_interface = pd.concat(
        [tx_meta_interface, permuted_tx_interface])
    # Merge with synthetic TX
    tx_meta_interface.rename({'cell':'cell_id'}, inplace=True, axis=1)
    synthetic_tx_meta = synthetic_txs.copy().drop('num',axis=1)
    # remove original TXs from original interface tumors
    tx_meta_interface = pd.concat([tx_meta_interface, synthetic_tx_meta])
    tx_meta_interface = GeoDataFrame(tx_meta_interface, geometry='point')
    # keep = ['gene', 'point', 'cell_id', 'tx_type', 'x', 'y', 'neighbor']
    # tx_meta_interface = tx_meta_interface[keep]
    # tx_meta_interface['point'] = points_from_xy(
    #     tx_meta_interface.x,tx_meta_interface.y)
    # tx_meta_interface.to_parquet(outfile, index=True)
    tx_meta_interface[['gene', 'x', 'y', 'cell_id', 'tx_type', 'neighbor']].to_csv(outfile, index=True)
    
    return tx_meta_interface

In [ ]:
def load_geo_h5ad(h5ad_filename, geo_parquet_filename):
    adata = sc.read_h5ad(h5ad_filename)
    geodata = gpd.read_parquet(geo_parquet_filename)
    index_name = adata.obs.index.name
    adata.obs = adata.obs.reset_index().merge(geodata, left_on=index_name, right_index=True, how='left').set_index(index_name)
    return adata

def write_geo_adata(adata, filename):
    adata_copy = adata.copy()
    geometry_columns = adata.obs.dtypes[adata.obs.dtypes == 'geometry'].index.tolist()
    adata_copy.obs = adata_copy.obs.drop(geometry_columns, axis=1)
    adata_copy.write_h5ad(filename)

In [ ]:
tx_meta = pd.read_csv('../data/merscope_hcc1/transcript_meta.csv', index_col=0)
adata_obj = load_geo_h5ad(os.path.join(MERSCOPE_DATADIR, 'working_anndata_tx_reassigned.h5ad'), os.path.join(MERSCOPE_DATADIR, 'cell_polygons.parquet'))
adata_obj.X = adata_obj.layers['count']

In [ ]:
cell_dist_synthetic, cell_meta_synthetic, mask_distance_synthetic, adata_synthetic, permuted_tx_interface = prepare_permutation_scheme(
    adata_obj, tx_meta,
)

In [ ]:
max_synth_txs = [5, 10, 15, 20, 25, 30]

for max_synth_tx in max_synth_txs:
    tx_meta_interface = generate_synthetic_dataset(
        cell_dist_synthetic, cell_meta_synthetic, mask_distance_synthetic, adata_synthetic, permuted_tx_interface,
        outfile=os.path.join(MERSCOPE_DATADIR, f'synthetic_tx{max_synth_tx:02d}_transcript_meta.csv'),
        max_noise_tx=max_synth_tx, n_points=1000, doublet_prob=0.5, tx2m_dist=1, deg_max_p=0.05, deg_min_effect=0.6,
        max_pct_tumor=0.2, min_pct_diff=0.2, max_pct_other=0.25, min_pct_tumor=0.5,
        seed=2
    )

In [ ]:
# save subset components

adata_subset = adata_obj[(adata_obj.obs['center_x'] <= 6000) & ((adata_obj.obs['center_y'] >= 6000))].copy()
write_geo_adata(adata_subset, os.path.join(MERSCOPE_DATADIR, 'adata_subset.h5ad'))

tx_meta_subset = tx_meta[(tx_meta['cell'].isin(adata_subset.obs_names))].copy()
tx_meta_subset.to_csv(os.path.join(MERSCOPE_DATADIR, 'transcript_meta_subset.csv'), index=True)

adata_subset.obs[['center_x', 'center_y', 'cell_type']].reset_index(names=['cell']).to_csv(
    os.path.join(MERSCOPE_DATADIR, 'cell_meta_subset.csv'), index=False
)

GeoDataFrame(adata_subset.obs.reset_index(names=['cell'])[['cell', 'polygon']], geometry='polygon').to_parquet(
    os.path.join(MERSCOPE_DATADIR, 'cell_boundary_polygons_subset.parquet'), index=False
)

# Alternative synthetic datasets (altering doublet parameter)

In [ ]:
max_synth_tx = 5
doublet_prob = 0.05
tx_meta_interface = generate_synthetic_dataset(
    cell_dist_synthetic, cell_meta_synthetic, mask_distance_synthetic, adata_synthetic, permuted_tx_interface,
    outfile=os.path.join(MERSCOPE_DATADIR, f'synthetic_tx{max_synth_tx:02d}_doublet{doublet_prob:.02f}_transcript_meta.csv'),
    max_noise_tx=max_synth_tx, n_points=1000, doublet_prob=doublet_prob, tx2m_dist=1, deg_max_p=0.05, deg_min_effect=0.6,
    max_pct_tumor=0.2, min_pct_diff=0.2, max_pct_other=0.25, min_pct_tumor=0.5,
    seed=2
)

In [ ]:
max_synth_tx = 5
doublet_prob = 0.95
tx_meta_interface = generate_synthetic_dataset(
    cell_dist_synthetic, cell_meta_synthetic, mask_distance_synthetic, adata_synthetic, permuted_tx_interface,
    outfile=os.path.join(MERSCOPE_DATADIR, f'synthetic_tx{max_synth_tx:02d}_doublet{doublet_prob:.02f}_transcript_meta.csv'),
    max_noise_tx=max_synth_tx, n_points=1000, doublet_prob=doublet_prob, tx2m_dist=1, deg_max_p=0.05, deg_min_effect=0.6,
    max_pct_tumor=0.2, min_pct_diff=0.2, max_pct_other=0.25, min_pct_tumor=0.5,
    seed=2
)